# Potato Leaf Disease Classification — Research-Grade Pipeline with EfficientNetB3

**Task:** 3-class classification of potato leaf imagery (*Early Blight*, *Late Blight*, *Healthy*) on the PLD dataset (256x256).

## 1. Motivation

Potato is a staple crop for over a billion people worldwide. Late blight (*Phytophthora infestans*), the pathogen behind the Irish famine, and early blight (*Alternaria solani*) remain responsible for significant yield losses annually. Manual field diagnosis is slow, error-prone, and inaccessible to many smallholder farmers. A reliable computer-vision classifier deployed on a cheap edge device could give farmers near-instant triage advice. This notebook builds a rigorous, reproducible deep-learning pipeline and evaluates it honestly on a **held-out test set that is never consulted during model development**.

## 2. Related Work

- **Hughes, D.P. & Salathé, M. (2015).** *An open access repository of images on plant health to enable the development of mobile disease diagnostics.* arXiv:1511.08060. Introduced PlantVillage, the canonical public plant-disease image corpus; this work established single-leaf, controlled-background classification as the standard benchmark task and demonstrated CNN feasibility for it.
- Mohanty et al. (2016, *Frontiers in Plant Science*) showed >99% accuracy on PlantVillage with AlexNet/GoogLeNet, while documenting large accuracy drops on real-field images — motivating careful augmentation and honest reporting here.
- **Tan & Le (2019).** *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML. Compound scaling of depth/width/resolution; EfficientNetB3 offers a strong accuracy/compute trade-off suited to Kaggle GPU budgets.
- Geifman & El-Yaniv (2017); Chawla et al. (2002, SMOTE) inform our treatment of stratification and class imbalance.
- Selvaraju et al. (2017). *Grad-CAM.* ICCAT — visual explanations used here to verify the model attends to lesion regions rather than background artifacts.

## 3. Methodology (summary)

| Component | Choice |
|---|---|
| Backbone | EfficientNetB3 (ImageNet pretrained), 2-phase head→fine-tune transfer |
| Input size | 256x256 (native PLD resolution) |
| Split | Stratified 80/10/10 over pooled images; **test set locked away before any development decision** |
| Imbalance | Inverse-frequency class weights in the loss |
| Augmentation | Horizontal/vertical flip, ±20° rotation, ±15% shift, 0.85–1.25 brightness, ≤0.2 zoom (justified below) |
| Loss | Categorical cross-entropy with 0.1 label smoothing |
| Callbacks | EarlyStopping on **val macro-F1**, ReduceLROnPlateau, best-only ModelCheckpoint |
| Metrics | Accuracy, macro-F1, per-class precision/recall/F1, confusion matrices, training curves |
| Interpretability | Grad-CAM overlays on test images |
| Ablation | EfficientNetB0 vs B3 under identical protocol |

## 4. Reproducibility

All sources of randomness (Python `hash`, `random`, NumPy, TensorFlow) are seeded; deterministic TF ops are enabled where supported. Results may vary marginally across hardware but the protocol is fully specified.

In [ ]:
# Step 1 — Imports, seeding, environment check
import os, random, json, warnings, collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 5. Data

### 5.1 Dataset & input paths

The PLD dataset (`PLD_3_Classes_256`) ships pre-sized 256x256 RGB leaf images in three classes. We resolve the mount location defensively because Kaggle occasionally exposes datasets under `/kaggle/input/<slug>` vs `/kaggle/input/datasets/<owner>/<slug>`.

### 5.2 Held-out test discipline

The raw archive contains vendor Train/Val/Test folders. To guarantee an untouched test partition, we **pool the Train+Validation images**, re-split them stratified into new train/validation partitions, and reserve the vendor Testing folder as the **held-out test set** — loaded once, only for final evaluation in Section 8. No augmentation, checkpoint selection, early stopping, or hyper-parameter choice ever sees it.

In [ ]:
# Step 2 — Paths, class names, hyperparameters
BASE = None
for cand in [
    Path("/kaggle/input/potato-disease-leaf-datasetpld/PLD_3_Classes_256"),
    Path("/kaggle/input/datasets/rizwan123456789/potato-disease-leaf-datasetpld/PLD_3_Classes_256"),
]:
    if cand.exists():
        BASE = cand; break
assert BASE is not None, f"Dataset not found; contents: {list(Path('/kaggle/input').rglob('*'))[:40]}"

TRAIN_DIR, VAL_DIR, TEST_DIR = BASE/"Training", BASE/"Validation", BASE/"Testing"
EXTS = {".jpg", ".jpeg", ".png"}

def collect(split_dir):
    rows = []
    for cls_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for p in sorted(cls_dir.rglob("*")):
            if p.suffix.lower() in EXTS:
                rows.append({"filepath": str(p), "label": cls_dir.name})
    return pd.DataFrame(rows)

df_train_raw = collect(TRAIN_DIR)
df_val_raw   = collect(VAL_DIR)
df_test      = collect(TEST_DIR)          # HELD OUT — final evaluation only
CLASS_NAMES  = sorted(df_train_raw.label.unique())
NUM_CLASSES  = len(CLASS_NAMES)

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Raw pools -> train {len(df_train_raw)}, val {len(df_val_raw)}, test(held-out) {len(df_test)}")

IMG_SIZE, CHANNELS = 256, 3
BATCH_SIZE    = 32
EPOCHS_HEAD   = 6
EPOCHS_FINE   = 14
LR_HEAD, LR_FINE = 1e-3, 5e-5
FINE_TUNE_AT  = 100
LABEL_SMOOTH  = 0.1
WORKING       = Path("/kaggle/working")

### 5.3 Class distribution

Explicit inspection of imbalance is required before choosing a mitigation. The bar chart below shows per-class counts in every partition.

In [ ]:
# Step 3 — Per-class counts (train/val/test)
counts = pd.DataFrame({
    s: d.label.value_counts().reindex(CLASS_NAMES)
    for s, d in [("Train", df_train_raw), ("Val", df_val_raw), ("Test (held-out)", df_test)]
})
print(counts.to_string(), "\nTotal:", counts.values.sum())

fig, ax = plt.subplots(figsize=(9, 5))
counts.plot(kind="bar", ax=ax, color=["#e74c3c", "#3498db", "#2ecc71"], edgecolor="white")
ax.set_title("Per-class image counts per partition", fontweight="bold")
ax.set_ylabel("# images"); ax.set_xlabel("")
for c in ax.containers:
    ax.bar_label(c, fontsize=9)
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(WORKING/"class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

imb_ratio = counts["Train"].max() / counts["Train"].min()
print(f"Imbalance ratio (max/min in training pool): {imb_ratio:.2f}x")

In [ ]:
# Step 4 — Sample images per class
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(14, 3*NUM_CLASSES))
for r, cls in enumerate(CLASS_NAMES):
    imgs = df_train_raw[df_train_raw.label == cls].sample(5, random_state=SEED).filepath.tolist()
    for c in range(5):
        ax = axes[r, c]; ax.imshow(mpimg.imread(imgs[c])); ax.axis("off")
        if c == 0:
            ax.set_title(cls, loc="left", fontweight="bold")
plt.suptitle("Representative training samples", y=1.01, fontweight="bold")
plt.tight_layout(); plt.savefig(WORKING/"samples_train.png", dpi=150, bbox_inches="tight"); plt.show()

### 5.4 Stratified re-split of the development pool

Train+Validation are pooled and re-partitioned 90/10 with stratification so both partitions preserve the class prior. The held-out test set stays exactly as delivered by the vendor.

### 5.5 Handling class imbalance

With Healthy under-represented (~0.63x the majority class), a plain CE loss biases the decision boundary toward majority classes. We apply **inverse-frequency class weights** (`sklearn.compute_class_weight('balanced')`) inside the loss, which equalises the effective contribution of each class without distorting the image distribution seen by the network. This is preferred over oversampling here because duplication on a small dataset raises overfitting risk.

### 5.6 Augmentation policy (justification)

| Transform | Range | Justification |
|---|---|---|
| Horizontal / vertical flip | p=0.5 | Leaf orientation is arbitrary in handheld photos; lesions can appear anywhere on the blade |
| Rotation | ±20° | Mimics camera tilt in the field |
| Width/height shift | ±15% | Simulates off-centre framing |
| Zoom | ≤0.2 | Varies apparent lesion scale |
| Brightness | 0.85–1.25 | Robustness to lighting; kept mild to avoid destroying colour cues that distinguish blight stages |

Deliberately excluded: aggressive colour-space jitter/hue shifts — hue carries diagnostic signal (chlorotic vs necrotic tissue) and corrupting it risks teaching colour-invariant features that blur disease classes. No augmentation is applied to validation/test data (evaluation must reflect deployment conditions).

In [ ]:
# Step 5 — Stratified split + class weights
dev_pool = pd.concat([df_train_raw, df_val_raw], ignore_index=True)
df_train, df_val = train_test_split(
    dev_pool, test_size=0.10, stratify=dev_pool.label, random_state=SEED
)
df_train, df_val = df_train.reset_index(drop=True), df_val.reset_index(drop=True)

for name, d in [("train", df_train), ("val", df_val), ("TEST (locked)", df_test)]:
    print(name.ljust(14), dict(sorted(collections.Counter(d.label).items())))

y_dev = df_train.label.map({c: i for i, c in enumerate(CLASS_NAMES)}).values
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_dev)
CLASS_WEIGHTS = {i: float(round(w, 4)) for i, w in enumerate(cw)}
print("\nClass weights:", CLASS_WEIGHTS)

LABEL_MAP = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in LABEL_MAP.items()}

In [ ]:
# Step 6 — Data generators (augmentation on train only)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15,
    horizontal_flip=True, vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.85, 1.25],
    fill_mode="nearest",
)
eval_datagen = ImageDataGenerator(rescale=1./255)

common = dict(target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode="categorical", color_mode="rgb")
train_gen = train_datagen.flow_from_dataframe(
    df_train, x_col="filepath", y_col="label", shuffle=True, seed=SEED, **common)
val_gen = eval_datagen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label", shuffle=False, **common)
test_gen = eval_datagen.flow_from_dataframe(
    df_test, x_col="filepath", y_col="label", shuffle=False, **common)

assert train_gen.class_indices == LABEL_MAP, train_gen.class_indices
STEPS = len(train_gen)

## 6. Model

Two-phase transfer learning:
1. **Phase 1 (head):** backbone frozen, train the GAP → BN → Dropout → Dense(256) → BN → Dropout → softmax head at LR 1e-3.
2. **Phase 2 (fine-tune):** unfreeze EfficientNetB3 layers ≥ index 100, train end-to-end at LR 5e-5 to adapt high-level features without catastrophic forgetting.

Regularisation: 0.1 label smoothing (Müller et al., 2019), dropout, light L2 on the penultimate dense. The primary model-selection metric is **validation macro-F1**, which is insensitive to class prevalence — appropriate given the imbalance.

We also register a streaming macro-F1 Keras metric so EarlyStopping/ReduceLR can monitor it natively.

In [ ]:
# Step 7 — Macro-F1 metric + model builder
class MacroF1(keras.metrics.Metric):
    """Streaming macro-F1 accumulated via a confusion matrix."""
    def __init__(self, num_classes, name="f1_macro", **kw):
        super().__init__(name=name, **kw)
        self.num_classes = num_classes
        self.cm = self.add_weight(name="cm", shape=(num_classes, num_classes),
                                  initializer="zeros", dtype=tf.float64)
    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.argmax(y_true, axis=-1)
        yp = tf.argmax(y_pred, axis=-1)
        cm = tf.cast(tf.math.confusion_matrix(yt, yp,
                     num_classes=self.num_classes, dtype=tf.int64), tf.float64)
        self.cm.assign_add(cm)
    def result(self):
        tp = tf.linalg.diag_part(self.cm)
        fp = tf.reduce_sum(self.cm, 0) - tp
        fn = tf.reduce_sum(self.cm, 1) - tp
        prec = tp / tf.maximum(tp + fp, 1e-12)
        rec  = tp / tf.maximum(tp + fn, 1e-12)
        f1 = 2 * prec * rec / tf.maximum(prec + rec, 1e-12)
        return tf.reduce_mean(f1)
    def reset_state(self):
        self.cm.assign(tf.zeros_like(self.cm))

METRICS = [keras.metrics.CategoricalAccuracy(name="accuracy"), MacroF1(NUM_CLASSES)]

def build_model(backbone_cls, num_classes, img_size, lr):
    base = backbone_cls(include_top=False, weights="imagenet",
                        input_shape=(img_size, img_size, 3))
    base.trainable = False
    inputs = keras.Input((img_size, img_size, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = Model(inputs, outputs, name=f"{backbone_cls.__name__}_PLD")
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=METRICS)
    return model, base

def get_callbacks(tag):
    return [
        EarlyStopping(monitor="val_f1_macro", mode="max", patience=6,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(WORKING/f"ckpt_{tag}.keras", monitor="val_f1_macro",
                        mode="max", save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                          min_lr=1e-7, verbose=1),
        CSVLogger(WORKING/f"log_{tag}.csv"),
    ]

model_b3, base_b3 = build_model(EfficientNetB3, NUM_CLASSES, IMG_SIZE, LR_HEAD)
model_b3.summary()

### 6.1 Phase 1 — classification head

In [ ]:
# Step 8 — Phase 1: head training (B3)
h_head = model_b3.fit(
    train_gen, epochs=EPOCHS_HEAD, validation_data=val_gen,
    class_weight=CLASS_WEIGHTS, callbacks=get_callbacks("b3_head"), verbose=1)
print(f"\nPhase 1 best val macro-F1: {max(h_head.history['val_f1_macro']):.4f}")

### 6.2 Phase 2 — fine-tuning top backbone layers

In [ ]:
# Step 9 — Phase 2: fine-tune (B3)
base_b3.trainable = True
for layer in base_b3.layers[:FINE_TUNE_AT]:
    layer.trainable = False
n_train = sum(l.trainable for l in base_b3.layers)
print(f"Fine-tuning layers {FINE_TUNE_AT}/{len(base_b3.layers)} "
      f"(trainable base layers: {n_train})")

model_b3.compile(optimizer=keras.optimizers.Adam(LR_FINE),
                 loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=METRICS)

h_fine = model_b3.fit(
    train_gen, epochs=EPOCHS_FINE, validation_data=val_gen,
    class_weight=CLASS_WEIGHTS, callbacks=get_callbacks("b3_fine"), verbose=1)
model_b3.load_weights(WORKING/"ckpt_b3_fine.keras")   # best-val-F1 weights
print(f"\nPhase 2 best val macro-F1: {max(h_fine.history['val_f1_macro']):.4f}")

## 7. Ablation — EfficientNetB0 vs EfficientNetB3

To quantify the value of compound scaling within a fixed budget, we repeat an abbreviated version of the identical protocol (same seeds, split, weights, augmentation, callbacks) with EfficientNetB0: 3 head epochs + 5 fine-tune epochs. This is a *cheap* ablation, not a full replication; conclusions about B0 should be read accordingly.

In [ ]:
# Step 10 — Ablation: EfficientNetB0
model_b0, base_b0 = build_model(EfficientNetB0, NUM_CLASSES, IMG_SIZE, LR_HEAD)
h0_head = model_b0.fit(train_gen, epochs=3, validation_data=val_gen,
                       class_weight=CLASS_WEIGHTS,
                       callbacks=get_callbacks("b0_head"), verbose=1)

base_b0.trainable = True
for layer in base_b0.layers[:FINE_TUNE_AT]:
    layer.trainable = False
model_b0.compile(optimizer=keras.optimizers.Adam(LR_FINE),
                 loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=METRICS)
h0_fine = model_b0.fit(train_gen, epochs=5, validation_data=val_gen,
                       class_weight=CLASS_WEIGHTS,
                       callbacks=get_callbacks("b0_fine"), verbose=1)
model_b0.load_weights(WORKING/"ckpt_b0_fine.keras")

val_gen.reset(); p_b3v = model_b3.predict(val_gen, verbose=0)
val_gen.reset(); p_b0v = model_b0.predict(val_gen, verbose=0)
y_val = val_gen.classes
ablation = pd.DataFrame([
    {"Backbone": "EfficientNetB0", "Params(M)": round(model_b0.count_params()/1e6, 1),
     "Val Acc": accuracy_score(y_val, p_b0v.argmax(1)),
     "Val macro-F1": f1_score(y_val, p_b0v.argmax(1), average="macro")},
    {"Backbone": "EfficientNetB3", "Params(M)": round(model_b3.count_params()/1e6, 1),
     "Val Acc": accuracy_score(y_val, p_b3v.argmax(1)),
     "Val macro-F1": f1_score(y_val, p_b3v.argmax(1), average="macro")},
]).set_index("Backbone")
print(ablation.to_string())
ablation.to_csv(WORKING/"ablation_effnet_b0_vs_b3.csv")

### 7.1 Training curves

In [ ]:
# Step 11 — Training curves
H = {}
for k in h_head.history:
    H[k] = h_head.history[k] + h_fine.history[k]
split_ep = len(h_head.history["loss"])

plots = [("accuracy", "Accuracy"), ("loss", "Loss"), ("f1_macro", "Macro-F1")]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (key, title) in zip(axes, plots):
    ax.plot(H[key], "-o", ms=3, label="train")
    ax.plot(H[f"val_{key}"], "-o", ms=3, label="val")
    ax.axvline(split_ep - 0.5, ls=":", color="gray", label="fine-tune starts")
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("epoch"); ax.legend()
plt.suptitle("EfficientNetB3 training dynamics", fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(WORKING/"training_curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Results on the held-out test set

The test generator was constructed up-front but its labels were never used until now. We report accuracy, macro-F1, per-class precision/recall/F1, and confusion matrices — both raw counts and row-normalised (error-mode view).

In [ ]:
# Step 12 — Final evaluation on held-out test set
test_gen.reset()
y_prob = model_b3.predict(test_gen, verbose=1)
y_pred = y_prob.argmax(1)
y_true = test_gen.classes

test_acc  = accuracy_score(y_true, y_pred)
test_f1m  = f1_score(y_true, y_pred, average="macro")
report    = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print(f"Test accuracy : {test_acc:.4f}\nTest macro-F1 : {test_f1m:.4f}\n")
print(report)

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Confusion matrix (counts)", fontweight="bold")
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd", xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Confusion matrix (row-normalised)", fontweight="bold")
for ax in axes: ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(WORKING/"confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

per_class = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                  output_dict=True)
pd.DataFrame(per_class).T.to_csv(WORKING/"per_class_metrics.csv")

In [ ]:
# Step 14 — FINAL CELL: save all artifacts to /kaggle/working/
ART = WORKING
model_b3.save(ART/"potato_effnetb3_best.keras")

with open(ART/"class_names.json", "w") as f:
    json.dump({
        "class_names": CLASS_NAMES,
        "label_map": LABEL_MAP,
        "img_size": IMG_SIZE,
        "seed": SEED,
    }, f, indent=2)

summary = f"""RESULTS SUMMARY — Potato Leaf Disease Classification (EfficientNetB3)
=================================================================
Seed                    : {SEED}
Input size              : {IMG_SIZE}x{IMG_SIZE}
Classes                 : {', '.join(CLASS_NAMES)}
Split                   : pooled train+val -> stratified 90/10; vendor Testing held out entirely
Imbalance handling      : inverse-frequency class weights {CLASS_WEIGHTS}
Augmentation            : flip H/V, rot +/-20deg, shift 15%, zoom <=0.2, brightness 0.85-1.25
Loss                    : categorical CE, label smoothing {LABEL_SMOOTH}
Selection metric        : val macro-F1 (EarlyStopping patience 6, best-only checkpoints)

Ablation (val set)      :
{ablation.to_string()}

HELD-OUT TEST METRICS (EfficientNetB3, best-val-F1 weights)
-----------------------------------------------------------
Test accuracy           : {test_acc:.4f}
Test macro-F1           : {test_f1m:.4f}

Per-class report:
{report}
"""
(ART/"results_summary.txt").write_text(summary)
print(summary)
print("\nArtifacts written:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:38s} {p.stat().st_size/1024:10.1f} KB")

## 9. Interpretability — Grad-CAM

Grad-CAM (Selvaraju et al., 2017) localises image regions driving the prediction. If the saliency concentrates on lesion tissue rather than background, we gain evidence the model uses disease-relevant features instead of dataset artifacts (e.g. background colour shortcuts). We overlay heatmaps for a few held-out test images, including any misclassifications when present.

In [ ]:
# Step 13 - Grad-CAM (Keras-3 safe: flat rebuild, nesting-proof, non-fatal)
import traceback
from tensorflow.keras import Input, Model as KModel

def _all_layers(model):
    out = []
    for l in model.layers:
        if isinstance(l, tf.keras.Model):
            out.extend(_all_layers(l))
        else:
            out.append(l)
    return out

def _owner_submodel(model, target):
    for l in model.layers:
        if l is target:
            return model
        if isinstance(l, tf.keras.Model):
            r = _owner_submodel(l, target)
            if r is not None:
                return r
    return None

def _find_last_conv_layer(model):
    convs = [l for l in _all_layers(model)
             if 'conv' in l.name.lower() and len(l.output.shape) == 4]
    return convs[-1] if convs else None

try:
    last_conv_layer = _find_last_conv_layer(model_b3)
    print("Last conv layer:", last_conv_layer.name if last_conv_layer else None)
    assert last_conv_layer is not None, "no 4D conv layer found"

    owner = _owner_submodel(model_b3, last_conv_layer) or model_b3
    # 1) conv feature extractor scoped INSIDE its owning submodel
    conv_extractor = KModel(owner.inputs, last_conv_layer.output)

    # 2) rebuild remaining outer tail as FLAT stack sharing the same weights
    tail, seen_owner = [], False
    for l in model_b3.layers:
        if l is owner:
            seen_owner = True
            continue
        if seen_owner and not isinstance(l, tf.keras.layers.InputLayer):
            tail.append(l)

    def _heatmap(img_array, pred_index=None):
        with tf.GradientTape() as tape:
            conv_out = conv_extractor(img_array, training=False)
            tape.watch(conv_out)
            x2 = conv_out
            for l in tail:
                x2 = l(x2)
            preds = x2
            if pred_index is None:
                pred_index = tf.argmax(preds[0])
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)
        pooled = tf.reduce_mean(grads, axis=(0, 1))
        heat = tf.nn.relu(conv_out[0] @ pooled[..., tf.newaxis])[..., 0]
        heat = heat / (tf.reduce_max(heat) + 1e-12)
        return heat.numpy()

    wrong_idx = np.where(y_pred != y_true)[0]
    show_rows = list(np.random.RandomState(SEED).choice(len(df_test), 4, replace=False))
    show_rows += list(wrong_idx[:2])

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, i in zip(axes.flat, show_rows[:6]):
        arr = img_to_array(load_img(df_test.filepath.iloc[i],
                                    target_size=(IMG_SIZE, IMG_SIZE))) / 255.0
        inp_arr = np.expand_dims(arr, 0)
        heat = _heatmap(inp_arr)
        heat = np.clip(heat, 0, 1)
        heat = np.array(Image.fromarray((heat*255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0
        ax.imshow(arr)
        ax.imshow(heat, cmap="jet", alpha=0.35, extent=[0, IMG_SIZE, IMG_SIZE, 0])
        ok = y_pred[i] == y_true[i]
        ax.set_title(f"true={df_test.label.iloc[i]} | pred={IDX_TO_CLASS[y_pred[i]]}"
                     f" {'OK' if ok else 'WRONG'}",
                     fontsize=10, color="#27ae60" if ok else "#c0392b", fontweight="bold")
        ax.axis("off")
    plt.suptitle("Grad-CAM overlays - held-out test images", fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.savefig(WORKING/"gradcam_examples.png", dpi=150, bbox_inches="tight"); plt.show()
except Exception:
    print("Grad-CAM skipped (NON-FATAL - artifact saving already completed above):")
    traceback.print_exc()


## 10. Limitations

1. **Domain gap:** PlantVillage-style corpora (including PLD) contain mostly single leaves on plain backgrounds; field photographs with clutter, occlusion, and variable illumination degrade accuracy substantially (Mohanty et al., 2016).
2. **Single dataset, single seed:** results are not averaged across seeds or cross-dataset validated; reported variance is therefore unknown.
3. **Coarse labels:** binary diseased/healthy per leaf ignores severity grading needed for dosage decisions.
4. **Cheap ablation:** B0 ran a shortened schedule, so the comparison favours B3's longer training as well as its capacity.
5. **No calibration analysis:** softmax confidence is not verified against true likelihoods (no ECE/reliability diagram).

## 11. Future Work

- Cross-validation with multiple seeds to report mean±std.
- Fine-grained severity regression and multi-leaf / whole-plant detection (object detection + classification).
- Domain adaptation / unlabeled field data (pseudo-labelling) to close the lab-to-field gap.
- Quantisation-aware training + pruning for on-device inference; conformal prediction for abstention on uncertain samples.
- Extend to other Solanaceae crops for a shared pathology embedding.

## 12. References

- Hughes, D.P. & Salathé, M. (2015). An open access repository of images on plant health... arXiv:1511.08060.
- Tan, M. & Le, Q.V. (2019). EfficientNet: Rethinking Model Scaling for CNNs. ICML.
- Mohanty, S.P., Hughes, D.P., Salathé, M. (2016). Using deep learning for image-based plant disease detection. Frontiers in Plant Science 7:1419.
- Selvaraju, R.R. et al. (2017). Grad-CAM: Visual Explanations from Deep Networks. ICCV.
- Müller, R., Kornblith, S., Hinton, G. (2019). When does label smoothing help? NeurIPS.
- Chawla, N.V. et al. (2002). SMOTE: Synthetic minority over-sampling technique. JAIR 16.